## DWPose vs MediaPipe on the h5 dataset

Compares MediaPipe and DWPose hand-landmark detection on `tmp/Testdata.h5`'s own
7-camera recording (`cam00`-`cam06`, 128 frames), which -- unlike the separate
`cam0`-`cam3` OAK rig used for `recordings/sign_capture_*` sessions -- has real,
matching ground-truth calibration (intrinsics *and* extrinsics) for this exact rig.

Structured in two parts:
1. **Part 1: 2D detection quality.** Raw extraction, detection-rate comparison,
   visual side-by-side, full-sequence grid videos, and the 2D-only mixup signals
   (cross-detector pixel disagreement, per-camera co-occurrence score) -- nothing
   here needs camera poses or triangulation. This is the current focus: getting
   the 2D detection/consistency right before trusting anything built on top of it.
2. **Part 2: 3D triangulation.** Everything that needs the real ground-truth
   extrinsics: naive and confidence-weighted triangulation, RANSAC camera-set
   selection + Kalman/RTS smoothing + jitter flagging, full body+hands
   triangulation, the geometric (leave-one-out) swap test, annotated grid videos,
   and the viser viewer.

**Undistortion**: `hand_multiview.triangulate_dlt`'s projection matrices
(`P = K @ [R|t]`) assume a pure pinhole model with no distortion term, but
MediaPipe/DWPose detect landmarks on the original, distorted images -- feeding
those coordinates in directly (as this notebook did before) is a real geometric
inconsistency, not just an approximation, and this rig's distortion is non-trivial
(rational model, k2/k4 up to ~70 on some cameras -- notably cam00, the same camera
where the mixup detectors below found confirmed swaps). Part 1's Step 3 corrects
for this via the new `hand_multiview.undistort_landmarks`, producing `_u`-suffixed
landmark dicts used by every quantitative comparison and all of Part 2 -- while
skeleton overlays drawn onto the actual (distorted) video frames keep using the
raw, un-undistorted landmarks, since undistorting the landmarks doesn't undistort
the image they're drawn on.

This notebook is exploratory -- reusable logic lives in `h5_dataset.py`,
`h5_hand_extraction.py`, `grid_video.py`, and additions to
`hand_multiview.py`/`dwpose_onnx.py`, so promoting whichever setup wins doesn't
require a rewrite. `h5_explore.ipynb` (unmodified) is the reference this borrows
its h5-access pattern from.

In [ ]:
import os
import random
import time as _time

import cv2
import numpy as np
import matplotlib.pyplot as plt

import hand_multiview as hmv
import h5_dataset as h5d
import h5_hand_extraction as hhe
import grid_video

H5_PATH = 'tmp/Testdata.h5'
DET_MODEL_PATH = 'models/dwpose/yolox_l.onnx'
POSE_MODEL_PATH = 'models/dwpose/dw-ll_ucoco_384.onnx'
CACHE_DIR = 'tmp/h5_extraction_cache'
MIN_CONFIDENCE = 0.5

## Part 1: 2D detection quality

### Step 1: calibration

Loaded early because Step 3's undistortion needs each camera's intrinsics
(`K`, `dist`) -- the extrinsics (`R`, `t`) it also provides aren't needed until
Part 2's triangulation.

In [ ]:
cal, camera_ids = hmv.load_chameleon_calibration(H5_PATH)
calib = hmv.load_all_camera_calibration(cal, camera_ids)
projection_matrices = {
    cam_id: hmv.build_projection_matrix(c['K'], c['R'], c['t'])
    for cam_id, c in calib.items()
}

# All cameras share one resolution in this dataset -- triangulate_sequence takes a
# single width/height rather than per-camera, so this only works because it's uniform.
width = calib[camera_ids[0]]['width']
height = calib[camera_ids[0]]['height']

print(f'{len(camera_ids)} cameras: {camera_ids}')
for cam_id in camera_ids:
    c = calib[cam_id]
    pos = -c['R'].T @ c['t']
    print(f"  {cam_id}: {c['width']:.0f}x{c['height']:.0f}  position={np.round(pos, 3)}")

### Step 2: extract hand+body landmarks (raw, distorted-image pixel space)

In [ ]:
frame_keys = h5d.discover_h5_frames(H5_PATH, camera_ids[0])
print(f'{len(frame_keys)} frames per camera')

(mp_landmarks_left, mp_confidence_left, mp_landmarks_right, mp_confidence_right,
 mp_landmarks_body, mp_confidence_body, mp_landmark_confidence_body) = (
    hhe.extract_mediapipe_hands_from_h5(H5_PATH, camera_ids, frame_keys, cache_dir=CACHE_DIR, force=False)
)
(dw_landmarks_left, dw_confidence_left, dw_landmark_confidence_left,
 dw_landmarks_right, dw_confidence_right, dw_landmark_confidence_right,
 dw_landmarks_body, dw_confidence_body, dw_landmark_confidence_body) = (
    hhe.extract_dwpose_hands_from_h5(
        H5_PATH, camera_ids, frame_keys, DET_MODEL_PATH, POSE_MODEL_PATH,
        cache_dir=CACHE_DIR, force=False,
    )
)

### Step 3: undistort landmarks

Produces the `_u`-suffixed landmark dicts used by every quantitative comparison
below and all of Part 2. Raw (non-`_u`) landmarks are kept around for drawing on
the actual distorted video frames (Steps 5-6, and the annotated grid videos in
Part 2) -- undistorting the *landmarks* doesn't undistort the *image* they get
drawn on, so using `_u` coordinates there would misalign the skeleton against the
visible pixels.

In [ ]:
mp_landmarks_left_u = hmv.undistort_landmarks(mp_landmarks_left, calib)
mp_landmarks_right_u = hmv.undistort_landmarks(mp_landmarks_right, calib)
mp_landmarks_body_u = hmv.undistort_landmarks(mp_landmarks_body, calib)
dw_landmarks_left_u = hmv.undistort_landmarks(dw_landmarks_left, calib)
dw_landmarks_right_u = hmv.undistort_landmarks(dw_landmarks_right, calib)
dw_landmarks_body_u = hmv.undistort_landmarks(dw_landmarks_body, calib)

print('Undistorted mp/dw hand+body landmarks -- used for all quantitative 2D/3D '
      'analysis below. Confidence dicts are untouched by undistortion (only x,y '
      'move), so the same *_confidence_* variables are reused as-is.')

### Step 4: detection-rate comparison

In [ ]:
def detection_rate_table(landmarks, confidence, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE):
    total = len(frame_keys)
    rows = {}
    for cam_id in camera_ids:
        n = sum(1 for fk in frame_keys if confidence.get(cam_id, {}).get(fk, 0) >= min_confidence)
        rows[cam_id] = (n, total)
    return rows


mp_left_rates = detection_rate_table(mp_landmarks_left, mp_confidence_left, camera_ids, frame_keys)
mp_right_rates = detection_rate_table(mp_landmarks_right, mp_confidence_right, camera_ids, frame_keys)
dw_left_rates = detection_rate_table(dw_landmarks_left, dw_confidence_left, camera_ids, frame_keys)
dw_right_rates = detection_rate_table(dw_landmarks_right, dw_confidence_right, camera_ids, frame_keys)


def fmt(rates, cam_id):
    n, total = rates[cam_id]
    return f'{n}/{total} ({100 * n / total:.0f}%)'


print(f"{'cam':<8} {'MP-left':<14} {'MP-right':<14} {'DW-left':<14} {'DW-right':<14}")
for cam_id in camera_ids:
    print(
        f"{cam_id:<8} {fmt(mp_left_rates, cam_id):<14} {fmt(mp_right_rates, cam_id):<14} "
        f"{fmt(dw_left_rates, cam_id):<14} {fmt(dw_right_rates, cam_id):<14}"
    )

### Step 5: visual side-by-side (green = MediaPipe, red = DWPose)

Pick whichever `SIDE` ('left'/'right') showed more consistent detections in the
table above. Drawn on the raw (distorted) frames using raw landmarks -- this is a
"does the detector look right" check, not a geometry check.

In [ ]:
SIDE = 'right'
mp_landmarks = mp_landmarks_right if SIDE == 'right' else mp_landmarks_left
mp_confidence = mp_confidence_right if SIDE == 'right' else mp_confidence_left
dw_landmarks = dw_landmarks_right if SIDE == 'right' else dw_landmarks_left
dw_confidence = dw_confidence_right if SIDE == 'right' else dw_confidence_left

candidates = [
    (cam_id, fk) for cam_id in camera_ids for fk in frame_keys
    if mp_confidence.get(cam_id, {}).get(fk, 0) >= MIN_CONFIDENCE
    and dw_confidence.get(cam_id, {}).get(fk, 0) >= MIN_CONFIDENCE
]
print(f'{len(candidates)} camera/frame pairs where both detectors are confident')

rng = random.Random(0)
samples = rng.sample(candidates, min(8, len(candidates)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (cam_id, fk) in zip(axes.ravel(), samples):
    img = None
    for _, imgs in h5d.iter_h5_frames(H5_PATH, [cam_id], [fk]):
        img = imgs.get(cam_id)
    if img is None:
        continue
    img = img.copy()
    h, w = img.shape[:2]
    mp_pixel_xy = hmv.landmark_dict_to_pixel_xy(mp_landmarks[cam_id][fk], w, h)
    dw_pixel_xy = hmv.landmark_dict_to_pixel_xy(dw_landmarks[cam_id][fk], w, h)
    hmv.draw_hand_skeleton(img, mp_pixel_xy, color=(0, 255, 0), point_color=(0, 200, 0))
    hmv.draw_hand_skeleton(img, dw_pixel_xy, color=(0, 0, 255), point_color=(0, 0, 200))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{cam_id} / {fk}')
    ax.axis('off')
for ax in axes.ravel()[len(samples):]:
    ax.axis('off')
fig.suptitle(f'{SIDE} hand: green=MediaPipe, red=DWPose')
fig.tight_layout()

### Step 6: full-sequence 2D grid videos (plain)

Full-sequence, all-camera view -- easier to spot per-frame failures (like the
frame 59/60/73/75 issues that motivated the mixup-detector work) than the 8-sample
grid above. No detector overlays yet (those need Part 2's outputs); re-rendered
with full annotations in Part 2's Step 13, overwriting these same files.

In [ ]:
GRID_VIDEO_DIR = 'tmp'
os.makedirs(GRID_VIDEO_DIR, exist_ok=True)

mp_grid_path = os.path.join(GRID_VIDEO_DIR, 'mediapipe_hand_grid.mp4')
dw_grid_path = os.path.join(GRID_VIDEO_DIR, 'dwpose_hand_grid.mp4')

print('Rendering MediaPipe grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, mp_grid_path,
    mp_landmarks_left, mp_confidence_left, mp_landmarks_right, mp_confidence_right,
    method_label='MediaPipe', min_confidence=MIN_CONFIDENCE,
    landmarks_body=mp_landmarks_body,
)

print('Rendering DWPose grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, dw_grid_path,
    dw_landmarks_left, dw_confidence_left, dw_landmarks_right, dw_confidence_right,
    method_label='DWPose', min_confidence=MIN_CONFIDENCE,
    landmark_confidence_left=dw_landmark_confidence_left, landmark_confidence_right=dw_landmark_confidence_right,
    landmarks_body=dw_landmarks_body,
)

### Step 7: 2D-only mixup signals

Two detectors that need only 2D landmarks, no triangulation -- kept here in Part 1
since they're pure 2D-detection-quality signals. The geometric (leave-one-out)
swap test needs triangulation and lives in Part 2's Step 12, using the candidate
pairs `flagged_pairs` below to avoid sweeping the expensive test over everything.

In [ ]:
# 7.1: cheap cross-detector 2D pre-filter (on undistorted coordinates, since this
# is a numeric pixel-distance comparison, not a drawing operation), swept over
# every camera/frame.
disagreements = {}
for cam_id in camera_ids:
    for fk in frame_keys:
        r = hmv.cross_detector_hand_disagreement(
            mp_landmarks_left_u, mp_landmarks_right_u, dw_landmarks_left_u, dw_landmarks_right_u,
            cam_id, fk, width, height,
        )
        if r is not None:
            disagreements.setdefault(cam_id, {})[fk] = r

flagged_pairs = sorted(
    (cam_id, fk) for cam_id, by_frame in disagreements.items()
    for fk, r in by_frame.items() if r['likely_swap']
)
print(f'{len(flagged_pairs)} camera/frame pairs flagged by the cross-detector pre-filter '
      f'(out of {sum(len(v) for v in disagreements.values())} checked)')
for cam_id, fk in flagged_pairs[:20]:
    r = disagreements[cam_id][fk]
    print(f"  {cam_id} / {fk}: same_side={r['same_side']:.1f}px cross_side={r['cross_side']:.1f}px")

In [ ]:
# 7.2: per-camera co-occurring-badness score -- formalizes the observation that
# started this work ("when one hand is bad for a camera/frame, the other often
# is too") as a point-biserial correlation between per-frame low-confidence flags
# for the left vs. right hand, per camera/detector. Confidence-based only, so
# raw vs. undistorted landmarks makes no difference here.
def co_occurrence_score(confidence_left, confidence_right, cam_id, frame_keys, min_confidence=MIN_CONFIDENCE):
    bad_left = np.array([
        1.0 if confidence_left.get(cam_id, {}).get(fk, 0) < min_confidence else 0.0 for fk in frame_keys
    ])
    bad_right = np.array([
        1.0 if confidence_right.get(cam_id, {}).get(fk, 0) < min_confidence else 0.0 for fk in frame_keys
    ])
    if bad_left.std() == 0 or bad_right.std() == 0:
        return float('nan')
    return float(np.corrcoef(bad_left, bad_right)[0, 1])


print(f"{'cam':<8} {'MP co-occurrence':<20} {'DW co-occurrence'}")
for cam_id in camera_ids:
    mp_score = co_occurrence_score(mp_confidence_left, mp_confidence_right, cam_id, frame_keys)
    dw_score = co_occurrence_score(dw_confidence_left, dw_confidence_right, cam_id, frame_keys)
    print(f"{cam_id:<8} {mp_score:<20.2f} {dw_score:.2f}")

## Part 2: 3D triangulation

Everything below needs the real ground-truth extrinsics loaded in Step 1, and
uses the undistorted (`_u`) landmarks from Step 3 -- not the raw ones, which are
only for drawing onto actual video frames.

### Step 8: naive triangulation comparison -- MediaPipe-only / DWPose-only / combined (max-confidence)

Uses the fixed ground-truth `projection_matrices` from Step 1 -- no pose
estimation.

In [ ]:
def build_variants(landmarks_mp, confidence_mp, landmarks_dw, confidence_dw):
    combined_landmarks, combined_confidence = hmv.combine_landmark_sources(
        [(landmarks_mp, confidence_mp), (landmarks_dw, confidence_dw)], mode='max_confidence',
    )
    return {
        'mediapipe': (landmarks_mp, confidence_mp),
        'dwpose': (landmarks_dw, confidence_dw),
        'combined_max_conf': (combined_landmarks, combined_confidence),
    }


variants = {
    'left': build_variants(mp_landmarks_left_u, mp_confidence_left, dw_landmarks_left_u, dw_confidence_left),
    'right': build_variants(mp_landmarks_right_u, mp_confidence_right, dw_landmarks_right_u, dw_confidence_right),
}

reconstructions = {}
for side, side_variants in variants.items():
    reconstructions[side] = {}
    for name, (landmarks, confidence) in side_variants.items():
        reconstructions[side][name] = hmv.triangulate_sequence(
            landmarks, projection_matrices, width, height,
            min_confidence=MIN_CONFIDENCE, confidence=confidence,
        )

In [ ]:
def wrist_jitter(reconstruction):
    """Mean frame-to-frame Euclidean displacement of the wrist landmark (id '0') --
    a simple proxy for reconstruction smoothness/quality; noisier 2D detections
    should triangulate to a jumpier 3D wrist even though the real wrist barely
    moves frame-to-frame at this capture rate.
    """
    keys = sorted(reconstruction.keys(), key=int)
    displacements = []
    prev = None
    for k in keys:
        wrist = reconstruction[k].get('0')
        if wrist is None:
            continue
        if prev is not None:
            displacements.append(float(np.linalg.norm(np.array(wrist) - np.array(prev))))
        prev = wrist
    return float(np.mean(displacements)) if displacements else float('nan')


print(f"{'side':<8} {'variant':<20} {'frames':<10} {'mean wrist jitter':<20}")
for side, side_recons in reconstructions.items():
    for name, recon in side_recons.items():
        print(f"{side:<8} {name:<20} {len(recon):<10} {wrist_jitter(recon):<20.5f}")

### Step 9: confidence-weighted triangulation using DWPose's per-landmark confidence

Combines both sources at the pixel level with `weighted_average` (so MediaPipe's
frame-level score and DWPose's per-landmark scores both influence the blended 2D
position), then triangulates with `triangulate_sequence_weighted`, using DWPose's
per-landmark confidence as the per-observation DLT weight. Compare its jitter
against the unweighted variants above.

In [ ]:
landmark_confidence_by_side = {'left': dw_landmark_confidence_left, 'right': dw_landmark_confidence_right}

reconstructions_weighted = {}
for side in ('left', 'right'):
    landmarks_mp, confidence_mp = variants[side]['mediapipe']
    landmarks_dw, confidence_dw = variants[side]['dwpose']
    landmark_confidence_dw = landmark_confidence_by_side[side]

    combined_landmarks, combined_confidence = hmv.combine_landmark_sources(
        [(landmarks_mp, confidence_mp), (landmarks_dw, confidence_dw)], mode='weighted_average',
    )
    reconstructions_weighted[side] = hmv.triangulate_sequence_weighted(
        combined_landmarks, projection_matrices, width, height, landmark_confidence_dw,
        min_confidence=MIN_CONFIDENCE, confidence=combined_confidence,
    )

print(f"{'side':<8} {'variant':<26} {'frames':<10} {'mean wrist jitter':<20}")
for side, side_recons in reconstructions.items():
    for name, recon in side_recons.items():
        print(f"{side:<8} {name:<26} {len(recon):<10} {wrist_jitter(recon):<20.5f}")
for side, recon in reconstructions_weighted.items():
    print(f"{side:<8} {'weighted_average+DWconf':<26} {len(recon):<10} {wrist_jitter(recon):<20.5f}")

### Step 10: RANSAC camera-set selection + Kalman/RTS smoothing + jitter flagging

The triangulations above use every confident camera per landmark -- a bad-but-
confident view (e.g. a hand mixup) still gets full geometric weight. This step
instead picks a geometrically-consistent inlier camera SET per frame first (RANSAC
over camera pairs on the wrist, reprojection-error inlier counting, hysteresis
against the previous frame's choice), then triangulates every landmark from just
that set. A constant-velocity Kalman filter + RTS backward smoother then gives a
reference trajectory per landmark, and frames where the raw RANSAC wrist deviates
sharply from it are flagged -- this is the principled version of "a wrist can't
move too much frame to frame." Ported/generalized from
`ransac_keypoint_reconstruction.ipynb` (validated on a different rig) into
`hand_multiview.py`.

In [ ]:
# Reprojection threshold re-derived empirically for this dataset's real
# calibration (expect far tighter residuals than a rough-PnP rig's ~50px) --
# check the printed mean_reproj_err values below and tighten/loosen as needed.
RANSAC_REPROJ_THRESH_PX = 15.0

camera_selections = {}
reconstructions_ransac = {}
reconstructions_smoothed = {}
jitter_flags = {}
jitter_residuals = {}

for side, side_variants in variants.items():
    camera_selections[side] = {}
    reconstructions_ransac[side] = {}
    reconstructions_smoothed[side] = {}
    jitter_flags[side] = {}
    jitter_residuals[side] = {}
    for name, (landmarks, confidence) in side_variants.items():
        selection = hmv.ransac_select_cameras_sequence(
            landmarks, confidence, projection_matrices, width, height,
            reproj_thresh_px=RANSAC_REPROJ_THRESH_PX, min_confidence=MIN_CONFIDENCE,
        )
        recon_ransac = hmv.triangulate_sequence_ransac(
            landmarks, projection_matrices, width, height, range(21), selection,
        )
        recon_frame_keys = sorted(recon_ransac.keys(), key=int)
        n_inliers_by_frame = {fk: sel['n_inliers'] for fk, sel in selection.items()}
        recon_smoothed = hmv.smooth_reconstruction_sequence(
            recon_ransac, recon_frame_keys, range(21), n_inliers_by_frame,
        )
        flags, residuals = hmv.flag_jitter_frames(recon_ransac, recon_smoothed, recon_frame_keys)

        camera_selections[side][name] = selection
        reconstructions_ransac[side][name] = recon_ransac
        reconstructions_smoothed[side][name] = recon_smoothed
        jitter_flags[side][name] = flags
        jitter_residuals[side][name] = residuals

print(f"{'side':<8} {'variant':<20} {'frames':<10} {'mean n_inliers':<16} {'mean reproj err (px)':<22} {'jitter-flagged'}")
for side, side_sel in camera_selections.items():
    for name, selection in side_sel.items():
        if not selection:
            print(f"{side:<8} {name:<20} {'0':<10} {'n/a':<16} {'n/a':<22} 0")
            continue
        n_inliers_vals = [s['n_inliers'] for s in selection.values()]
        err_vals = [s['mean_reproj_err'] for s in selection.values()]
        n_flagged = len(jitter_flags[side][name])
        print(
            f"{side:<8} {name:<20} {len(selection):<10} {np.mean(n_inliers_vals):<16.2f} "
            f"{np.mean(err_vals):<22.2f} {n_flagged}"
        )

### Step 10b: per-camera RANSAC reprojection error

The summary table above only reports the mean error of whichever cameras RANSAC
actually chose as inliers each frame -- it doesn't show which specific cameras
tend to disagree. This reprojects each frame's already-triangulated wrist into
*every* confident camera (via the new `hmv.ransac_per_camera_reproj_error`), so a
camera that's a chronic outlier (e.g. cam00, which has by far the strongest lens
distortion of the 7) shows up as a wide/high box here even on frames where RANSAC
excluded it from the winning set.

In [ ]:
per_camera_reproj_err = {}
for side, side_variants in variants.items():
    per_camera_reproj_err[side] = {}
    for name, (landmarks, confidence) in side_variants.items():
        per_camera_reproj_err[side][name] = hmv.ransac_per_camera_reproj_error(
            landmarks, confidence, reconstructions_ransac[side][name], projection_matrices, width, height,
            min_confidence=MIN_CONFIDENCE,
        )

variant_names = ['mediapipe', 'dwpose', 'combined_max_conf']
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
for row, side in enumerate(('left', 'right')):
    for col, name in enumerate(variant_names):
        ax = axes[row, col]
        errors = per_camera_reproj_err[side][name]
        data = [errors.get(cam_id, {}).values() for cam_id in camera_ids]
        data = [list(v) if v else [np.nan] for v in data]
        ax.boxplot(data, tick_labels=camera_ids, showfliers=True)
        ax.set_title(f'{side} / {name}')
        ax.tick_params(axis='x', rotation=45)
        if col == 0:
            ax.set_ylabel('reproj err (px)')
fig.suptitle('RANSAC per-camera reprojection error (wrist), all cameras')
fig.tight_layout()

print(f"{'side':<8} {'variant':<20} {'cam':<8} {'mean (px)':<12} {'median (px)':<12} {'n frames'}")
for side in ('left', 'right'):
    for name in variant_names:
        errors = per_camera_reproj_err[side][name]
        for cam_id in camera_ids:
            vals = list(errors.get(cam_id, {}).values())
            if not vals:
                print(f"{side:<8} {name:<20} {cam_id:<8} {'n/a':<12} {'n/a':<12} 0")
                continue
            print(f"{side:<8} {name:<20} {cam_id:<8} {np.mean(vals):<12.2f} {np.median(vals):<12.2f} {len(vals)}")

### Step 11: full body + hands triangulation

Both extractors already compute body keypoints for free (MediaPipe's Pose call
was already running for the wrist-proximity check; DWPose's body slice is part of
the same forward pass) -- `h5_hand_extraction.py` returns/caches them too,
remapped to a shared COCO-17 schema (`BLAZEPOSE_TO_COCO17`) so MediaPipe's and
DWPose's body output can be combined/triangulated exactly like the hands above.
This gives the viewer the whole signer figure per variant instead of two floating
hand clouds, and an anatomical sanity check (hand-wrist vs. body-wrist distance,
Step 12) since both models estimate the wrist position independently.

In [ ]:
body_variants = build_variants(mp_landmarks_body_u, mp_confidence_body, dw_landmarks_body_u, dw_confidence_body)

reconstructions_body = {}
for name, (landmarks, confidence) in body_variants.items():
    reconstructions_body[name] = hmv.triangulate_sequence(
        landmarks, projection_matrices, width, height,
        min_confidence=MIN_CONFIDENCE, confidence=confidence, landmark_ids=range(17),
    )

print(f"{'variant':<20} {'frames':<10} {'body joints (frame 0)'}")
for name, recon in reconstructions_body.items():
    keys = sorted(recon.keys(), key=int)
    n_joints = len(recon[keys[0]]) if keys else 0
    print(f"{name:<20} {len(recon):<10} {n_joints}")

### Step 12: geometric mixup detectors -- LOO swap test + body-wrist consistency

**12.0 runs first**: a purely 2D, single-camera/frame test using each
detector's OWN body-pose wrist keypoints as a same-frame reference for which
hand is which (`hmv.detect_body_wrist_hand_swap`) -- no other camera or the
other detector needs a confident detection that frame, so it's the cheapest
and least conditional of these tests, and fixes what it can before the
pricier multiview/cross-detector tests run on what's left (see
`hand_multiview.py`'s mixup-detection section comment).

The leave-one-out (LOO) swap test needs triangulation (it compares each camera's
own observation against a multiview consensus built from every *other* camera),
so it lives here rather than with Step 7's 2D-only signals. Run on Step 7's
`flagged_pairs` plus explicitly the four motivating frames (DWPose right bad at
59/60, MediaPipe right bad at 73/75), so the hypothesis that started this work
gets a direct geometric verdict.

Note 12.0's `BWRIST` and 12.2's `BODYWRIST` are two different tests despite the
similar name: 12.0 is a 2D per-camera/frame swap test (compares this camera's
own hand-wrist pixel to its own body-wrist pixel); 12.2 is a 3D, whole-sequence
consistency check (compares the RANSAC-triangulated hand-wrist trajectory to
the triangulated body-wrist trajectory via median+MAD).

In [ ]:
# 12.0: body-wrist proximity swap test (2D-only, first-pass -- see markdown above).
# Cheap enough to run over every camera/frame pair directly, no curated subset
# needed (unlike 12.1's LOO test, which needs other confident cameras).
MOTIVATING_FRAME_INDICES = [59, 60, 73, 75]
motivating_frame_keys = [frame_keys[i] for i in MOTIVATING_FRAME_INDICES if i < len(frame_keys)]

BODYWRIST_SWAP_MARGIN_PX = 20.0

mp_bodywrist_swap = hmv.detect_body_wrist_hand_swaps_sequence(
    mp_landmarks_left_u, mp_landmarks_right_u, mp_landmarks_body_u, camera_ids, frame_keys, width, height,
    swap_margin_px=BODYWRIST_SWAP_MARGIN_PX,
)
dw_bodywrist_swap = hmv.detect_body_wrist_hand_swaps_sequence(
    dw_landmarks_left_u, dw_landmarks_right_u, dw_landmarks_body_u, camera_ids, frame_keys, width, height,
    swap_margin_px=BODYWRIST_SWAP_MARGIN_PX,
)


def summarize_bodywrist(results, label):
    n_checked = sum(len(by_frame) for by_frame in results.values())
    n_swaps = sum(1 for by_frame in results.values() for r in by_frame.values() if r['swap_detected'])
    print(f'{label}: {n_swaps}/{n_checked} (camera, frame) pairs flagged as body-wrist swaps')


summarize_bodywrist(mp_bodywrist_swap, 'MediaPipe')
summarize_bodywrist(dw_bodywrist_swap, 'DWPose')

print('\nVerdict on the four motivating frames:')
for idx in MOTIVATING_FRAME_INDICES:
    if idx >= len(frame_keys):
        continue
    fk = frame_keys[idx]
    for cam_id in camera_ids:
        for label, results in (('MediaPipe', mp_bodywrist_swap), ('DWPose', dw_bodywrist_swap)):
            r = results.get(cam_id, {}).get(fk)
            if r is None:
                continue
            verdict = 'SWAP' if r['swap_detected'] else 'ok'
            print(f"  idx={idx} ({fk}) / {cam_id} / {label}: {verdict} (margin={r['margin']:.1f}px)")


In [ ]:
# 12.1: leave-one-out swap-hypothesis test.
candidate_pairs = set(flagged_pairs) | {
    (cam_id, fk) for cam_id in camera_ids for fk in motivating_frame_keys
}


def run_swap_detection(landmarks_left, confidence_left, landmarks_right, confidence_right, pairs):
    results = {}
    for cam_id, fk in pairs:
        r = hmv.detect_camera_hand_swap(
            landmarks_left, confidence_left, landmarks_right, confidence_right,
            projection_matrices, width, height, cam_id, fk, min_confidence=MIN_CONFIDENCE,
        )
        if r is not None:
            results.setdefault(cam_id, {})[fk] = r
    return results


mp_swap_results = run_swap_detection(
    mp_landmarks_left_u, mp_confidence_left, mp_landmarks_right_u, mp_confidence_right, candidate_pairs,
)
dw_swap_results = run_swap_detection(
    dw_landmarks_left_u, dw_confidence_left, dw_landmarks_right_u, dw_confidence_right, candidate_pairs,
)


def summarize_swaps(results, label):
    n_checked = sum(len(by_frame) for by_frame in results.values())
    n_swaps = sum(1 for by_frame in results.values() for r in by_frame.values() if r['swap_detected'])
    print(f'{label}: {n_swaps}/{n_checked} candidate (camera, frame) pairs confirmed as swaps')


print(f'LOO swap test run on {len(candidate_pairs)} candidate (camera, frame) pairs')
summarize_swaps(mp_swap_results, 'MediaPipe')
summarize_swaps(dw_swap_results, 'DWPose')

print('\nVerdict on the four motivating frames:')
for idx in MOTIVATING_FRAME_INDICES:
    if idx >= len(frame_keys):
        continue
    fk = frame_keys[idx]
    for cam_id in camera_ids:
        for label, results in (('MediaPipe', mp_swap_results), ('DWPose', dw_swap_results)):
            r = results.get(cam_id, {}).get(fk)
            if r is None:
                continue
            verdict = 'SWAP' if r['swap_detected'] else 'ok'
            print(f"  idx={idx} ({fk}) / {cam_id} / {label}: {verdict} (margin={r['margin']:.1f}px)")

In [ ]:
# 12.2: hand-wrist vs. body-wrist consistency. Reuses hmv.flag_jitter_frames's
# median+MAD pattern by wrapping each side's hand-wrist and body-wrist
# trajectories into single-landmark reconstructions.
BODY_WRIST_COCO = {'left': 9, 'right': 10}  # COCO-17: left_wrist=9, right_wrist=10

body_hand_consistency_flags = {}
body_hand_consistency_residuals = {}
for side in ('left', 'right'):
    body_wrist_key = str(BODY_WRIST_COCO[side])
    for variant in ('mediapipe', 'dwpose', 'combined_max_conf'):
        hand_recon = reconstructions_ransac[side][variant]
        body_recon = reconstructions_body[variant]
        common_frames = sorted(set(hand_recon) & set(body_recon), key=int)
        hand_wrist_only = {fk: {'0': hand_recon[fk]['0']} for fk in common_frames if '0' in hand_recon[fk]}
        body_wrist_only = {
            fk: {'0': body_recon[fk][body_wrist_key]} for fk in common_frames if body_wrist_key in body_recon[fk]
        }
        flags, residuals = hmv.flag_jitter_frames(
            hand_wrist_only, body_wrist_only, common_frames, reference_landmark_id=0,
        )
        body_hand_consistency_flags[(side, variant)] = flags
        body_hand_consistency_residuals[(side, variant)] = residuals

print(f"{'side':<8} {'variant':<20} {'frames compared':<18} {'flagged':<10} {'median dist (m)'}")
for (side, variant), residuals in body_hand_consistency_residuals.items():
    flags = body_hand_consistency_flags[(side, variant)]
    med = np.median(list(residuals.values())) if residuals else float('nan')
    print(f"{side:<8} {variant:<20} {len(residuals):<18} {len(flags):<10} {med:.3f}")

### Step 13: re-render the grid videos with detector overlays

Re-renders the same `mediapipe_hand_grid.mp4` / `dwpose_hand_grid.mp4` from Step
6, now that Steps 10-12 have all the detector output available: each camera slot
gets the triggered per-camera detectors (cross-detector pre-filter `XDET`, LOO
swap test `LOO` -- both concern the left/right pair as a whole, so no hand suffix)
burned into its bottom-left corner, and the two leftover grid slots (7 cameras in
a 3x3 grid) show the config used and the frame-level `JITTER`/`BODYWRIST` flags
(per hand) for the frame currently playing. The LOO swap test is re-run here over
every camera/frame (Step 12.1 only checked a curated candidate set) so the video
reflects the true per-frame status everywhere. Rendering itself still draws on
raw (distorted) frames using raw landmarks -- only the detector *flags* come from
the undistorted-based Part 2 computations.

In [ ]:
# Full LOO swap sweep (all camera/frame pairs) -- Step 12.1 only checked a
# curated candidate set; the video should show the true per-frame status
# everywhere it's watched.
mp_swap_full = hmv.detect_camera_hand_swaps_sequence(
    mp_landmarks_left_u, mp_confidence_left, mp_landmarks_right_u, mp_confidence_right,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)
dw_swap_full = hmv.detect_camera_hand_swaps_sequence(
    dw_landmarks_left_u, dw_confidence_left, dw_landmarks_right_u, dw_confidence_right,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)


def _bool_view(results, key='swap_detected'):
    return {cam_id: {fk: r[key] for fk, r in by_frame.items()} for cam_id, by_frame in results.items()}


xdet_bool = _bool_view(disagreements, key='likely_swap')


def build_global_flags(variant):
    return {
        'JITTER': {'left': jitter_flags['left'][variant], 'right': jitter_flags['right'][variant]},
        'BODYWRIST': {
            'left': body_hand_consistency_flags[('left', variant)],
            'right': body_hand_consistency_flags[('right', variant)],
        },
    }


thresholds_text_common = [
    f'MIN_CONFIDENCE={MIN_CONFIDENCE}',
    f'RANSAC_REPROJ_THRESH_PX={RANSAC_REPROJ_THRESH_PX}',
]
mp_camera_stats = [
    f'{cam_id}: {co_occurrence_score(mp_confidence_left, mp_confidence_right, cam_id, frame_keys):.2f}'
    for cam_id in camera_ids
]
dw_camera_stats = [
    f'{cam_id}: {co_occurrence_score(dw_confidence_left, dw_confidence_right, cam_id, frame_keys):.2f}'
    for cam_id in camera_ids
]

print('Rendering annotated MediaPipe grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, mp_grid_path,
    mp_landmarks_left, mp_confidence_left, mp_landmarks_right, mp_confidence_right,
    method_label='MediaPipe', min_confidence=MIN_CONFIDENCE,
    per_camera_flags={'XDET': xdet_bool, 'LOO': _bool_view(mp_swap_full), 'BWRIST': _bool_view(mp_bodywrist_swap)},
    global_flags=build_global_flags('mediapipe'),
    thresholds_text=thresholds_text_common,
    camera_stats_text=['MP co-occ:'] + mp_camera_stats,
    landmarks_body=mp_landmarks_body,
)

print('Rendering annotated DWPose grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, dw_grid_path,
    dw_landmarks_left, dw_confidence_left, dw_landmarks_right, dw_confidence_right,
    method_label='DWPose', min_confidence=MIN_CONFIDENCE,
    per_camera_flags={'XDET': xdet_bool, 'LOO': _bool_view(dw_swap_full), 'BWRIST': _bool_view(dw_bodywrist_swap)},
    global_flags=build_global_flags('dwpose'),
    thresholds_text=thresholds_text_common,
    camera_stats_text=['DW co-occ:'] + dw_camera_stats,
    landmark_confidence_left=dw_landmark_confidence_left, landmark_confidence_right=dw_landmark_confidence_right,
    landmarks_body=dw_landmarks_body,
)

### Step 13b: corrected 2D landmarks -- flip correction + velocity-based confidence penalty

Three pieces, all operating on the RAW (drawing-space) landmarks so the result
can go straight into a "corrected" grid video for a direct visual check:

1. **Stage 1 -- body-wrist flip correction** (`hmv.apply_swap_corrections` using
   Step 12.0's `mp_bodywrist_swap`/`dw_bodywrist_swap`): applied FIRST, ahead of
   the LOO fix, because it's the cheapest and least conditional of the detectors
   (needs only that camera's own body pose that frame, not any other camera or
   the other detector) -- see the ordering rationale in `hand_multiview.py`'s
   mixup-detection section comment.
2. **Stage 2 -- LOO flip correction** (`hmv.apply_swap_corrections` using a LOO
   sweep **re-run on the stage-1 output**, not Step 13's `mp_swap_full`/
   `dw_swap_full`, which was computed on the pre-stage-1 data and would be stale
   here now that some assignments have already changed): wherever the LOO test
   still finds a swap after stage 1, swap that entry's left/right landmarks *and*
   confidence between the two dicts.
3. **Velocity-based confidence penalty**: a 2D-only detector,
   `hmv.detect_high_velocity_frames`, flags any (camera, frame) where a hand's
   wrist moves more than `HIGH_VELOCITY_THRESH_PX` (200px/frame) -- physically
   implausible at this capture rate, so it shouldn't keep dominating a
   confidence-weighted combination. `hmv.apply_confidence_penalty` multiplies
   that (camera, frame, hand)'s confidence by `VELOCITY_CONFIDENCE_PENALTY`
   (1/100) rather than excluding it outright.

**Order matters** at every stage: each fix should only ever see the *previous*
stage's already-corrected data, never the original. A genuine swap corrupts the
trajectory right at the swap frame, producing an apparent velocity spike that has
nothing to do with real motion -- correcting swaps first (stages 1 and 2) means
that spike is gone before the velocity detector (stage 3) ever sees it, so a
validly-corrected frame isn't penalized for a problem that's already fixed.

**Known simplification, worth revisiting**: the penalty is applied to each
hand's whole-frame `confidence_left`/`confidence_right` score, not to DWPose's
per-landmark confidence dict -- the per-landmark scores are left untouched, so a
weighted triangulation using them wouldn't yet see this penalty.

Renders `mediapipe_hand_grid_corrected.mp4` / `dwpose_hand_grid_corrected.mp4`
with `BWRIST`/`LOO` (whether each stage's re-run detector still finds anything
after its own correction) and `HIVEL_L`/`HIVEL_R` flags -- compare against the
Step 13 videos to check whether the strategy actually cleans up the frames it's
supposed to, and whether it introduces new problems.


In [ ]:
HIGH_VELOCITY_THRESH_PX = 200.0
VELOCITY_CONFIDENCE_PENALTY = 0.01

# --- Stage 1: body-wrist flip correction (Step 12.0's mp_bodywrist_swap /
# dw_bodywrist_swap) -- applied first, see Step 13b markdown for why. ---
(mp_landmarks_left_bw, mp_confidence_left_bw,
 mp_landmarks_right_bw, mp_confidence_right_bw, mp_n_bodywrist_corrected) = hmv.apply_swap_corrections(
    mp_landmarks_left, mp_confidence_left, mp_landmarks_right, mp_confidence_right,
    _bool_view(mp_bodywrist_swap),
)
(dw_landmarks_left_bw, dw_confidence_left_bw,
 dw_landmarks_right_bw, dw_confidence_right_bw, dw_n_bodywrist_corrected) = hmv.apply_swap_corrections(
    dw_landmarks_left, dw_confidence_left, dw_landmarks_right, dw_confidence_right,
    _bool_view(dw_bodywrist_swap),
)
print(f'Stage 1 (body-wrist) corrected {mp_n_bodywrist_corrected} pairs for MediaPipe, '
      f'{dw_n_bodywrist_corrected} for DWPose')

# --- Stage 2: LOO flip correction, re-run on the stage-1 output (mp_swap_full /
# dw_swap_full from Step 13 was computed on the PRE-stage-1 data and would be
# stale as an input here). ---
mp_landmarks_left_bw_u = hmv.undistort_landmarks(mp_landmarks_left_bw, calib)
mp_landmarks_right_bw_u = hmv.undistort_landmarks(mp_landmarks_right_bw, calib)
dw_landmarks_left_bw_u = hmv.undistort_landmarks(dw_landmarks_left_bw, calib)
dw_landmarks_right_bw_u = hmv.undistort_landmarks(dw_landmarks_right_bw, calib)

mp_swap_stage2 = hmv.detect_camera_hand_swaps_sequence(
    mp_landmarks_left_bw_u, mp_confidence_left_bw, mp_landmarks_right_bw_u, mp_confidence_right_bw,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)
dw_swap_stage2 = hmv.detect_camera_hand_swaps_sequence(
    dw_landmarks_left_bw_u, dw_confidence_left_bw, dw_landmarks_right_bw_u, dw_confidence_right_bw,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)

(mp_landmarks_left_c, mp_confidence_left_c,
 mp_landmarks_right_c, mp_confidence_right_c, mp_n_corrected) = hmv.apply_swap_corrections(
    mp_landmarks_left_bw, mp_confidence_left_bw, mp_landmarks_right_bw, mp_confidence_right_bw,
    _bool_view(mp_swap_stage2),
)
(dw_landmarks_left_c, dw_confidence_left_c,
 dw_landmarks_right_c, dw_confidence_right_c, dw_n_corrected) = hmv.apply_swap_corrections(
    dw_landmarks_left_bw, dw_confidence_left_bw, dw_landmarks_right_bw, dw_confidence_right_bw,
    _bool_view(dw_swap_stage2),
)
print(f'Stage 2 (LOO, re-run on stage-1 output) corrected {mp_n_corrected} more pairs for MediaPipe, '
      f'{dw_n_corrected} for DWPose')

# --- Stage 3: velocity-based confidence penalty, recomputed on the fully
# (stage-1 + stage-2) corrected sequence -- see markdown above for why this
# ordering matters. ---
mp_velocity_left = hmv.detect_high_velocity_frames(
    mp_landmarks_left_c, camera_ids, frame_keys, width, height, velocity_thresh_px=HIGH_VELOCITY_THRESH_PX,
)
mp_velocity_right = hmv.detect_high_velocity_frames(
    mp_landmarks_right_c, camera_ids, frame_keys, width, height, velocity_thresh_px=HIGH_VELOCITY_THRESH_PX,
)
dw_velocity_left = hmv.detect_high_velocity_frames(
    dw_landmarks_left_c, camera_ids, frame_keys, width, height, velocity_thresh_px=HIGH_VELOCITY_THRESH_PX,
)
dw_velocity_right = hmv.detect_high_velocity_frames(
    dw_landmarks_right_c, camera_ids, frame_keys, width, height, velocity_thresh_px=HIGH_VELOCITY_THRESH_PX,
)


def _flagged_bool(velocity):
    return {cam_id: {fk: v['flagged'] for fk, v in by_frame.items()} for cam_id, by_frame in velocity.items()}


mp_confidence_left_final, n1 = hmv.apply_confidence_penalty(
    mp_confidence_left_c, _flagged_bool(mp_velocity_left), VELOCITY_CONFIDENCE_PENALTY,
)
mp_confidence_right_final, n2 = hmv.apply_confidence_penalty(
    mp_confidence_right_c, _flagged_bool(mp_velocity_right), VELOCITY_CONFIDENCE_PENALTY,
)
dw_confidence_left_final, n3 = hmv.apply_confidence_penalty(
    dw_confidence_left_c, _flagged_bool(dw_velocity_left), VELOCITY_CONFIDENCE_PENALTY,
)
dw_confidence_right_final, n4 = hmv.apply_confidence_penalty(
    dw_confidence_right_c, _flagged_bool(dw_velocity_right), VELOCITY_CONFIDENCE_PENALTY,
)

print(f"{'source':<20} {'penalized (>200px/f)'}")
print(f"{'MP left':<20} {n1}")
print(f"{'MP right':<20} {n2}")
print(f"{'DW left':<20} {n3}")
print(f"{'DW right':<20} {n4}")


In [ ]:
mp_landmarks_left_c_u = hmv.undistort_landmarks(mp_landmarks_left_c, calib)
mp_landmarks_right_c_u = hmv.undistort_landmarks(mp_landmarks_right_c, calib)
dw_landmarks_left_c_u = hmv.undistort_landmarks(dw_landmarks_left_c, calib)
dw_landmarks_right_c_u = hmv.undistort_landmarks(dw_landmarks_right_c, calib)

disagreements_corrected = {}
for cam_id in camera_ids:
    for fk in frame_keys:
        r = hmv.cross_detector_hand_disagreement(
            mp_landmarks_left_c_u, mp_landmarks_right_c_u, dw_landmarks_left_c_u, dw_landmarks_right_c_u,
            cam_id, fk, width, height,
        )
        if r is not None:
            disagreements_corrected.setdefault(cam_id, {})[fk] = r
n_flagged_corrected = sum(1 for by_frame in disagreements_corrected.values() for r in by_frame.values() if r['likely_swap'])
print(f'Cross-detector pre-filter after correction: {n_flagged_corrected} pairs still flagged '
      f'(was {len(flagged_pairs)} before correction)')

mp_swap_corrected = hmv.detect_camera_hand_swaps_sequence(
    mp_landmarks_left_c_u, mp_confidence_left_c, mp_landmarks_right_c_u, mp_confidence_right_c,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)
dw_swap_corrected = hmv.detect_camera_hand_swaps_sequence(
    dw_landmarks_left_c_u, dw_confidence_left_c, dw_landmarks_right_c_u, dw_confidence_right_c,
    projection_matrices, width, height, camera_ids, frame_keys, min_confidence=MIN_CONFIDENCE,
)
n_mp_swap_corrected = sum(1 for by_frame in mp_swap_corrected.values() for r in by_frame.values() if r['swap_detected'])
n_dw_swap_corrected = sum(1 for by_frame in dw_swap_corrected.values() for r in by_frame.values() if r['swap_detected'])
print(f'LOO swap test after correction: MediaPipe {n_mp_swap_corrected} still flagged, '
      f'DWPose {n_dw_swap_corrected} still flagged')

In [ ]:
mp_grid_path_corrected = os.path.join(GRID_VIDEO_DIR, 'mediapipe_hand_grid_corrected.mp4')
dw_grid_path_corrected = os.path.join(GRID_VIDEO_DIR, 'dwpose_hand_grid_corrected.mp4')

corrected_thresholds_text = [
    f'MIN_CONFIDENCE={MIN_CONFIDENCE}',
    f'BODYWRIST_SWAP_MARGIN_PX={BODYWRIST_SWAP_MARGIN_PX}',
    f'HIGH_VELOCITY_THRESH_PX={HIGH_VELOCITY_THRESH_PX}',
    f'VELOCITY_CONF_PENALTY={VELOCITY_CONFIDENCE_PENALTY}',
]

print('Rendering corrected MediaPipe grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, mp_grid_path_corrected,
    mp_landmarks_left_c, mp_confidence_left_final, mp_landmarks_right_c, mp_confidence_right_final,
    method_label='MediaPipe (corrected)', min_confidence=MIN_CONFIDENCE,
    per_camera_flags={
        'XDET': _bool_view(disagreements_corrected, key='likely_swap'),
        'LOO': _bool_view(mp_swap_corrected),
        'BWRIST': _bool_view(mp_bodywrist_swap),
        'HIVEL_L': _flagged_bool(mp_velocity_left),
        'HIVEL_R': _flagged_bool(mp_velocity_right),
    },
    thresholds_text=corrected_thresholds_text,
    camera_stats_text=[f'stage1 bodywrist: {mp_n_bodywrist_corrected}', f'stage2 loo: {mp_n_corrected}'],
    draw_velocity_circles=False,
    landmarks_body=mp_landmarks_body,
)

print('Rendering corrected DWPose grid video...')
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, dw_grid_path_corrected,
    dw_landmarks_left_c, dw_confidence_left_final, dw_landmarks_right_c, dw_confidence_right_final,
    method_label='DWPose (corrected)', min_confidence=MIN_CONFIDENCE,
    per_camera_flags={
        'XDET': _bool_view(disagreements_corrected, key='likely_swap'),
        'LOO': _bool_view(dw_swap_corrected),
        'BWRIST': _bool_view(dw_bodywrist_swap),
        'HIVEL_L': _flagged_bool(dw_velocity_left),
        'HIVEL_R': _flagged_bool(dw_velocity_right),
    },
    thresholds_text=corrected_thresholds_text,
    camera_stats_text=[f'stage1 bodywrist: {dw_n_bodywrist_corrected}', f'stage2 loo: {dw_n_corrected}'],
    draw_velocity_circles=False,
    landmarks_body=dw_landmarks_body,
)

### Step 14: viser -- color-coded variants, full body + hands, flagged frames highlighted

Each variant (mediapipe / dwpose / combined_max_conf) renders in a distinct color
as the primary visual cue (the X-axis offset stays as a secondary spatial
declutter, not the only distinguishing signal), showing the full body + both
hands per variant instead of two floating hand clouds, using the RANSAC-based
reconstructions from Steps 10/11. Frames flagged by `flag_jitter_frames` render in
red regardless of variant color. Reuses `h5_explore.ipynb`'s Y-up-to-Z-up
conversion (this rig's calibration world is Y-up).

In [ ]:
import viser
from viser import transforms as tf

# This rig's calibration world is Y-up (cal.gravity_world == [0, 1, 0]); viser
# defaults to Z-up -- flip so the figure is upright, matching h5_explore.ipynb.
R_WORLD_TO_VISER = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, -1.0, 0.0],
])


def to_viser_pose(R_wc, t_wc):
    T_cw = np.eye(4)
    T_cw[:3, :3] = R_wc.T
    T_cw[:3, 3] = -R_wc.T @ t_wc
    T_cw[:3, :3] = R_WORLD_TO_VISER @ T_cw[:3, :3]
    T_cw[:3, 3] = R_WORLD_TO_VISER @ T_cw[:3, 3]
    wxyz = tf.SO3.from_matrix(T_cw[:3, :3]).wxyz
    return wxyz, T_cw[:3, 3]


def to_viser_points(frame_points):
    return {lm_id: (R_WORLD_TO_VISER @ np.array(xyz)).tolist() for lm_id, xyz in frame_points.items()}


server = viser.ViserServer()
server.scene.set_up_direction('+z')
server.gui.configure_theme(dark_mode=True)

for cam_id, c in calib.items():
    wxyz, position = to_viser_pose(c['R'], c['t'])
    fov = 2.0 * np.arctan(c['height'] / (2.0 * c['K'][1, 1]))
    aspect = float(c['width'] / c['height'])
    server.scene.add_camera_frustum(
        f'/cameras/{cam_id}', fov=float(fov), aspect=aspect, scale=0.15,
        color=(80, 160, 255), wxyz=wxyz, position=position,
    )

print(f'Viser running at http://localhost:{server.get_port()}')
print('Showing mediapipe / dwpose / combined_max_conf: full body + both hands per '
      'variant, color-coded, offset along X so they do not overlap. Jitter-flagged '
      'frames (Step 10) render in red regardless of variant color.')

VARIANT_COLORS = {
    'mediapipe': (100, 200, 255),
    'dwpose': (255, 120, 120),
    'combined_max_conf': (160, 255, 160),
}
OFFSETS = {'mediapipe': -0.3, 'dwpose': 0.0, 'combined_max_conf': 0.3}

frame_keys_recon = sorted(
    {fk for side_recons in reconstructions_ransac.values() for recon in side_recons.values() for fk in recon}
    | {fk for recon in reconstructions_body.values() for fk in recon},
    key=int,
)
frame_slider = server.gui.add_slider('Frame', min=0, max=max(len(frame_keys_recon) - 1, 0), step=1, initial_value=0)
stop_button = server.gui.add_button('Stop viewer')
running = {'value': True}


@stop_button.on_click
def _(_):
    running['value'] = False


try:
    while running['value']:
        idx = frame_slider.value
        if frame_keys_recon:
            fk = frame_keys_recon[idx]
            for variant, color in VARIANT_COLORS.items():
                offset = np.array([OFFSETS[variant], 0, 0])

                body_points = reconstructions_body.get(variant, {}).get(fk, {})
                body_viser = {
                    lm_id: (np.array(xyz) + offset).tolist()
                    for lm_id, xyz in to_viser_points(body_points).items()
                }
                hmv.add_skeleton_frame(
                    server, body_viser, name_prefix=f'/{variant}/body',
                    connections=hmv.BODY_CONNECTIONS_COCO, point_color=color, bone_color=color,
                )

                for side in ('left', 'right'):
                    hand_points = reconstructions_ransac.get(side, {}).get(variant, {}).get(fk, {})
                    hand_viser = {
                        lm_id: (np.array(xyz) + offset).tolist()
                        for lm_id, xyz in to_viser_points(hand_points).items()
                    }
                    flagged = fk in jitter_flags.get(side, {}).get(variant, set())
                    hmv.add_skeleton_frame(
                        server, hand_viser, name_prefix=f'/{variant}/hand_{side}',
                        connections=hmv.HAND_CONNECTIONS, point_color=color, bone_color=color,
                        flagged=flagged,
                    )
        _time.sleep(0.05)
except KeyboardInterrupt:
    print('Viewer interrupted.')
finally:
    server.stop()
    print('Viser stopped.')

### Findings and recommendation

*Fill in from the actual detection-rate table, jitter comparison, mixup-detector
output, and viser inspection above:*

- Which detector had better/more consistent detection per camera?
- Did `combined_max_conf` beat both individual detectors, or just match the better one?
- Did per-landmark-confidence weighting reduce jitter relative to the unweighted combination?
- Did RANSAC camera-set selection (Step 10) reduce jitter relative to the naive
  all-cameras triangulation (Step 8)?
- Did undistorting the landmarks (Step 3) measurably reduce reprojection error /
  jitter relative to using raw distorted coordinates -- particularly on cam00,
  which has by far the strongest distortion of the 7 cameras?
- **Did the mixup hypothesis hold?** On frames 59/60 (DWPose right) and 73/75
  (MediaPipe right): did the cross-detector pre-filter and/or the LOO swap test
  (Step 7.1/12.1) confirm a camera-level hand-identity swap, or point to a plain
  low-confidence detection instead?
- Did hand-wrist vs. body-wrist consistency (12.2) or the per-camera
  co-occurrence score (7.2) corroborate which camera(s) are systematic mixup
  sources, if any?
- Recommendation for what to promote into a real pipeline (e.g. adapting
  `h5_explore.ipynb`'s own triangulation/viewer cells, or a new notebook) --
  not decided here, left for after running this once against the real data.

### Playground: tune detector thresholds and regenerate a grid video quickly

Edit the constants in the next cell and re-run it to iterate on detector
thresholds without re-running the whole notebook above. Recomputes the
cross-detector pre-filter, LOO swap test, RANSAC camera-set selection + jitter
flagging, and body-wrist consistency fresh from these values (on undistorted
landmarks) for whichever `PLAYGROUND_METHOD` you pick, then renders one annotated
grid video (drawn on raw frames/landmarks, as always) to
`tmp/{method}_hand_grid_playground.mp4` -- a separate file from the canonical
videos in Step 13, so iterating here never clobbers those.

In [ ]:
# --- Tune these and re-run this cell ---
PLAYGROUND_METHOD = 'mediapipe'  # 'mediapipe' or 'dwpose'
PLAYGROUND_MIN_CONFIDENCE = 0.5
PLAYGROUND_RANSAC_REPROJ_THRESH_PX = 15.0
PLAYGROUND_SWAP_MARGIN_PX = 5.0
PLAYGROUND_BODYWRIST_MARGIN_PX = 20.0
PLAYGROUND_SWAP_RATIO_THRESH = 0.5
PLAYGROUND_MIN_LOO_CAMERAS = 2
PLAYGROUND_MAD_FACTOR = 4.0
# ----------------------------------------

if PLAYGROUND_METHOD == 'mediapipe':
    p_landmarks_left_u, p_confidence_left = mp_landmarks_left_u, mp_confidence_left
    p_landmarks_right_u, p_confidence_right = mp_landmarks_right_u, mp_confidence_right
    p_landmarks_body_u, p_confidence_body = mp_landmarks_body_u, mp_confidence_body
    p_landmarks_left_raw, p_landmarks_right_raw = mp_landmarks_left, mp_landmarks_right
    p_landmarks_body_raw = mp_landmarks_body
    p_landmark_confidence_left, p_landmark_confidence_right = None, None
else:
    p_landmarks_left_u, p_confidence_left = dw_landmarks_left_u, dw_confidence_left
    p_landmarks_right_u, p_confidence_right = dw_landmarks_right_u, dw_confidence_right
    p_landmarks_body_u, p_confidence_body = dw_landmarks_body_u, dw_confidence_body
    p_landmarks_left_raw, p_landmarks_right_raw = dw_landmarks_left, dw_landmarks_right
    p_landmarks_body_raw = dw_landmarks_body
    p_landmark_confidence_left, p_landmark_confidence_right = dw_landmark_confidence_left, dw_landmark_confidence_right

# Cross-detector pre-filter always compares mp vs dw regardless of the primary
# method chosen above -- it's inherently a two-detector comparison. On
# undistorted coordinates, like Step 7.1.
p_xdet = {}
for cam_id in camera_ids:
    for fk in frame_keys:
        r = hmv.cross_detector_hand_disagreement(
            mp_landmarks_left_u, mp_landmarks_right_u, dw_landmarks_left_u, dw_landmarks_right_u,
            cam_id, fk, width, height, swap_ratio_thresh=PLAYGROUND_SWAP_RATIO_THRESH,
        )
        if r is not None:
            p_xdet.setdefault(cam_id, {})[fk] = r['likely_swap']

p_bwrist_swap = hmv.detect_body_wrist_hand_swaps_sequence(
    p_landmarks_left_u, p_landmarks_right_u, p_landmarks_body_u, camera_ids, frame_keys, width, height,
    swap_margin_px=PLAYGROUND_BODYWRIST_MARGIN_PX,
)
p_bwrist_bool = {
    cam_id: {fk: r['swap_detected'] for fk, r in by_frame.items()} for cam_id, by_frame in p_bwrist_swap.items()
}

p_swap_full = hmv.detect_camera_hand_swaps_sequence(
    p_landmarks_left_u, p_confidence_left, p_landmarks_right_u, p_confidence_right,
    projection_matrices, width, height, camera_ids, frame_keys,
    min_confidence=PLAYGROUND_MIN_CONFIDENCE, min_loo_cameras=PLAYGROUND_MIN_LOO_CAMERAS,
    swap_margin_px=PLAYGROUND_SWAP_MARGIN_PX,
)
p_swap_bool = {
    cam_id: {fk: r['swap_detected'] for fk, r in by_frame.items()} for cam_id, by_frame in p_swap_full.items()
}


def compute_side_pipeline(landmarks, confidence):
    selection = hmv.ransac_select_cameras_sequence(
        landmarks, confidence, projection_matrices, width, height,
        reproj_thresh_px=PLAYGROUND_RANSAC_REPROJ_THRESH_PX, min_confidence=PLAYGROUND_MIN_CONFIDENCE,
    )
    recon = hmv.triangulate_sequence_ransac(landmarks, projection_matrices, width, height, range(21), selection)
    recon_frame_keys = sorted(recon.keys(), key=int)
    n_inliers_by_frame = {fk: sel['n_inliers'] for fk, sel in selection.items()}
    smoothed = hmv.smooth_reconstruction_sequence(recon, recon_frame_keys, range(21), n_inliers_by_frame)
    flags, _ = hmv.flag_jitter_frames(recon, smoothed, recon_frame_keys, mad_factor=PLAYGROUND_MAD_FACTOR)
    return recon, flags


p_recon_left, p_jitter_left = compute_side_pipeline(p_landmarks_left_u, p_confidence_left)
p_recon_right, p_jitter_right = compute_side_pipeline(p_landmarks_right_u, p_confidence_right)

p_body_selection = hmv.ransac_select_cameras_sequence(
    p_landmarks_body_u, p_confidence_body, projection_matrices, width, height,
    reproj_thresh_px=PLAYGROUND_RANSAC_REPROJ_THRESH_PX, min_confidence=PLAYGROUND_MIN_CONFIDENCE,
)
p_body_recon = hmv.triangulate_sequence_ransac(
    p_landmarks_body_u, projection_matrices, width, height, range(17), p_body_selection,
)


def body_hand_flags(hand_recon, body_wrist_key):
    common = sorted(set(hand_recon) & set(p_body_recon), key=int)
    hand_only = {fk: {'0': hand_recon[fk]['0']} for fk in common if '0' in hand_recon[fk]}
    body_only = {fk: {'0': p_body_recon[fk][body_wrist_key]} for fk in common if body_wrist_key in p_body_recon[fk]}
    flags, _ = hmv.flag_jitter_frames(
        hand_only, body_only, common, reference_landmark_id=0, mad_factor=PLAYGROUND_MAD_FACTOR,
    )
    return flags


p_bodywrist_left = body_hand_flags(p_recon_left, str(BODY_WRIST_COCO['left']))
p_bodywrist_right = body_hand_flags(p_recon_right, str(BODY_WRIST_COCO['right']))

playground_output_path = f'tmp/{PLAYGROUND_METHOD}_hand_grid_playground.mp4'
grid_video.render_hand_grid_video(
    H5_PATH, camera_ids, frame_keys, playground_output_path,
    p_landmarks_left_raw, p_confidence_left, p_landmarks_right_raw, p_confidence_right,
    method_label=f'{PLAYGROUND_METHOD} (playground)', min_confidence=PLAYGROUND_MIN_CONFIDENCE,
    per_camera_flags={'XDET': p_xdet, 'LOO': p_swap_bool, 'BWRIST': p_bwrist_bool},
    global_flags={
        'JITTER': {'left': p_jitter_left, 'right': p_jitter_right},
        'BODYWRIST': {'left': p_bodywrist_left, 'right': p_bodywrist_right},
    },
    thresholds_text=[
        f'METHOD={PLAYGROUND_METHOD}',
        f'MIN_CONFIDENCE={PLAYGROUND_MIN_CONFIDENCE}',
        f'RANSAC_REPROJ_PX={PLAYGROUND_RANSAC_REPROJ_THRESH_PX}',
        f'SWAP_MARGIN_PX={PLAYGROUND_SWAP_MARGIN_PX}',
        f'BODYWRIST_MARGIN_PX={PLAYGROUND_BODYWRIST_MARGIN_PX}',
        f'SWAP_RATIO_THRESH={PLAYGROUND_SWAP_RATIO_THRESH}',
        f'MIN_LOO_CAMERAS={PLAYGROUND_MIN_LOO_CAMERAS}',
        f'MAD_FACTOR={PLAYGROUND_MAD_FACTOR}',
    ],
    camera_stats_text=[],
    landmark_confidence_left=p_landmark_confidence_left, landmark_confidence_right=p_landmark_confidence_right,
    landmarks_body=p_landmarks_body_raw,
)
print(f'Saved playground video to {playground_output_path}')